# Activity 3: Equation of State Fitting

In this notebook we will fit different EOS formulations to real data. This activity has the same goal of providing direct contact with real data. More specific to EOS fitting, through this notebook you will learn the following concepts:
- evaluate the quality of a formulation to describe data
- chose the formulation that describes data the best
- be aware of the uncertainty in thermoelastic parameters

In this notebook, we will fit the $P-\rho-T$ data of pyrolite from Solomatova & Caracas 2021 (https://doi.org/10.1029/2020JB021045). Pyrolite corresponds to the estimated bulk composition of Earth's mantle and has the chemical formula $\ce{0.5Na2O,2CaO,1.5Al2O3,4FeO,30MgO,24SiO2}$ (153 atoms). This data was computed from first-principles molecular dynamics simulations. This means solving the timle-dependent Schrödinger equation at the atomic level and calculate the global (thermodynamic) properties from atomic interactions.

<font color='red'>Cells with coding tasks are preceeded by text in red.</font>

## A foreword about numerical efficiency

This was intended to be a comment, but I ended up writing so much stuff that it's better to put it here. This section justifies my syntax, but has zero incidence on this class. You can skip this entirely if you want, but if you read it I hope you find it interesting.

Let me make a digression here about numerical considerations.
When defining the BM2 EOS, I could write <br>`(rho/rho_0)**(7.0/3.0)-(rho/rho_0)**(5.0/3.0)` or <br>`x=rho/rho_0 ; (x)**(7.0/3.0)-(x)**(5.0/3.0)` or <br>`(x)**7-(x)**5` or even <br>`x**5*(x**2-1)`.

When writing resource-intensive codes (runtime > 1 min) it is interesting to think about optimization. This mainly applies to low-level languages (C, Fortran, etc.), and is optimizer-dependent, but it is a good mental exercise to do here. At the hardware level, numbers are written in binary, and the CPU only understand instructions such as flipping a bit, moving it, or applying logic gates (and, or, xor...), meaning that doing an addition requires to **wire** the CPU in a specific way so that all the instructions reproduce the behavior of the mathematical operation we want. For additions it's easy, but for more complex operations it takes much more space. In modern CPUs, additions are cheap, multiplication is almost as cheap, but there is no hardware doing "division", so the division has to be done in several addition+multiplication steps, and therefore is MUCH more expensive. In old CPUs, divisions were ~100 times slower, so old programmers have PTSD over replacing `/2` by `*0.5`. Because I learned programming in Fortran, I also took the habit of replacing `*3/2`by `*1.5`, because it completely avoids a division. In modern CPUs divisions are faster but still more expensive. Following the same logic, evaluating exp, ln, or sine is ABSOLUTE HORROR. It is not possible to "wire" an exponential with CPU logic gates, so it is necessary to express exp as a combination of additions, multiplications, and divisions. How? With our good old friend Taylor expansion... This means to calculate exp(0.1) the computer needs to calculate 1 + 0.1 + 0.1^2/2 + 0.1^3/6 + ... until precision is good enough. Of course there are tricks (e.g. for a sine function, you only need the interval 0-pi/2 and then you apply symetries), but when using double precision you need to evaluate enough terms to reach the required precision (double precision = 10^-16 = 0.000000000000001). This is absolutely insane.

Naturally, physicists don't think about that. The first reason is because modern compilers do some work for us. For example, if you write `3/2` most compilers will replace it by `1.5`. This won't speed up your code by 500%, but at least it speeds up obvious parts. However, the second reason is physicists don't do that is because they don't know better. Let me tell you two stories when that actually mattered a lot.

Back when I was a PhD student, I was working with my good friend and colleague Dr. Hugo Vivien. He was a master student working on the interior structure model that I still use today (written in fortran, so very sensitive to management of memory and operations). After just 1 month of work, he identified 1) huge redundancies in various calculations (calculations that were done multiple times, although we didn't need to) and 2) an architecture that was not adapted to the logic employed. A code that was "working just fine" was in fact poorly optimized. After 1 other month of work he fixed these issues, which sped up the code by a factor of ~50. A code that used to run in 1-2 minutes was now running in 1-2 seconds. Dr. Vivien obtained his PhD at LAM, Marseille studying machine learning, and is currently a postdoc at LAM working on PLATO.

My dear friend and colleague Dr. Steven Rendon-Restrepo does 3D simulations of planet formation. As I told you in class, doubling resolution in 3D increases the number of cells by $2^3 = 8$, and if the complexity of the code is $\mathcal{O}(n^2)$ (which is considered efficient) then it's still 64 times slower. Of course this type of code is parallelized, but for many various reasons the speed up is not necessarily proportional to the number of processors: 100 times more CPU doesn't mean 100 times faster code. It depends on many things. During his PhD, Dr. Rendon-Restrepo was going as far as managing the time of propagation of information in the wires of the CPU so that he could speed up the code.

Obviously, we do what we can with the time and ressources that we have, but in my opinion the most inspiring part is that both Hugo and Steven did this on their own initiative. Nobody asked them to do it, and nobody even **thought** about this, and nobody cared if this was possible or necessary. But they did it, and their contributions are now part of history.

Back to our problem, if you look at how I define the BM2 equation of state:
- First, it is faster to compute and store x = rho/rho_0 once, and use it multiple times later.
- The next thing is that integers powers are cheaper to compute than float powers. `x**7` can be calculated as 6 multiplications. `x**7.0` will actually compute `exp(7.0*ln(x))`, meaning that we switch from simple CPU functions to not one but two insanely slow functions (ln and exp). Of course, both expressions will give the same result (not to numerical precision, but close enough). Your compiler *may* understand that `x**7 = x**7.0`, but it is not guaranteed.
- Lastly, following the idea of minimizing the number of multiplications, computing `(x)**7-(x)**5` does 6+4 multiplications = 10 multiplications. The most efficient way of calculating it would be:
```
x2 = x * x
x5 = x2 * x2 * x
y  = (x2 - 1) * x5
```
Which uses 4 multiplications. At this point it's overkill for our needs, but now you see what is the standard imposed on computer engineers. If you tell OpenAI that you can train their model for 900 million dollars instead of 1 billion dollar, they are going to be very happy (and they will give you a \\$10.000 bonus for saving them \\$100.000.000).

In [ ]:
import numpy as np
import pandas as pd
import astropy.constants as ac
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
def plt_scat_error(x,y,yerr,color,*plt_args):
    '''
    "errorbar" plot points with error bars.
    "scatter" plot points and each has an individual color.
    Matplotlib does not have a function doing both.
    Because of that, we define this function, doing exactly that.
    '''
    try:
        _ = yerr.shape
    except:
        yerr = 0*x
    try:
        _ = color.shape
    except:
        color = 0*x
    plt.errorbar(x,
            y,
            yerr=yerr,
            ls='',ecolor='k',
            marker='o', mfc='k',
            mec='k', ms=1, mew=4,zorder=5)
    plt.scatter(x,
            y,
            c=color,
            edgecolors='k',marker="o",ls="",zorder=10)

## Step 1: Get the data

The data is not available in open access, but authors provide it upon request. I have obtained a copy of the data some time ago. Load the data and plot it.

In [ ]:
pyrolite_data = pd.read_csv("SC21_pyrolite_pvt.dat",
                      delimiter=r'\s+',skiprows=1,
                      header=None,names=["a","V","rho-gcc","P-GPa","dP-GPa","T","dT"])

# ---- Print ----
print(pyrolite_data)

# ---- Plot ----
plt_scat_error(pyrolite_data["rho-gcc"],
               pyrolite_data["P-GPa"],
               pyrolite_data["dP-GPa"],
               pyrolite_data["T"])

plt.xlabel(r"Density [g.cm$^{-3}$]")
plt.ylabel("Pressure [GPa]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(6,4)
plt.colorbar(label="Temperature [K]")
plt.show()

## Step 2: fit the "cold" EOS

As you become acquainted with fitting procedures, let's combine the sub-steps of defining the function, evaluating parameters initial guess, eventual bounds, fitting, printing results, and visual confirmation.

First, we fit the data at the reference temperature $T_0=2000$ K, corresponding to the "cold" EOS.

<font color='red'>Task: complete the cell below so that the fit is successful </font>

In [ ]:
# -------------------------------------------------------
# 0) Define function to fit
# -------------------------------------------------------

def func_BM2(rho,rho_0,K_0):
    x = (rho/rho_0)**(1.0/3.0)
    return 1.5 * K_0 * ((x)**7-(x)**5)

# -------------------------------------------------------
# 1) Initial guess
# -------------------------------------------------------

rho_0_init = 
K_0_init = 

p_init = [
]

# -------------------------------------------------------
# 2) Fit
# -------------------------------------------------------

# keeping only data at T=2000K, mask that keeps values at 2000 +- 20 K.
mask_T0 = np.abs(pyrolite_data["T"]-2000) < 20
x = pyrolite_data["rho-gcc"][mask_T0]
y = pyrolite_data["P-GPa"][mask_T0]
yerr = pyrolite_data["dP-GPa"][mask_T0]

param_opt_bm2, pcov_bm2 = curve_fit(
    func_BM2,
    x,
    y,
    p0=p_init,
    sigma=yerr,
    absolute_sigma=True,
)

param_err_bm2 = np.sqrt(np.diag(pcov_bm2))

# -------------------------------------------------------
# 3) Print
# -------------------------------------------------------
param_names_bm2 = ["rho_0", "K_0"]

param_df_bm2 = pd.DataFrame({
    "Parameter": param_names_bm2,
    "Initial guess": p_init,
    "Best fit": param_opt_bm2,
    "1σ uncertainty": param_err_bm2
})

display(param_df_bm2)

# -------------------------------------------------------
# 3) Plot
# -------------------------------------------------------
# nothing to do here

# generate the best fit model on a nice uniform grid
rho_model = np.linspace(x.min(),x.max(),500)
P_model = func_BM2(rho_model, *param_opt_bm2)

# residuals computed at data points
residuals_bm2 = y - func_BM2(x, *param_opt_bm2)

# ---- plotting ----
plt.plot(rho_model,P_model)

plt.errorbar(x,y,yerr=yerr,marker="+",c="orange",ls="")

plt.xlabel(r"Density [g.cm$^{-3}$]")
plt.ylabel("Pressure [GPa]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(6,4)
plt.show()

# ---- Residuals ----
plt.errorbar(x,residuals_bm2,yerr=yerr,marker="+",c="orange",ls="")
plt.xlabel(r"Density [g.cm$^{-3}$]")
plt.ylabel("Pressure [GPa]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(6,4)
plt.show()

<font color='red'>Task: duplicate this cell for 3 more "cold" EOS formulations</font>

## Step 3: determine the best model

To determine the best model, we need to calculate the reduced chi-square $\chi_\nu^2 = \chi^2/\nu$ for each fit, where $\nu$ is the number of parameters and $\chi^2 = \sum_i \frac{(y_\mathrm{data}-y_\mathrm{model})^2}{\sigma_\mathrm{data}^2}$. The best model has the lowest $\chi_\nu^2$ value.

<font color='red'>Task: compute $\chi_\nu^2$ for each "cold" model, put them in a table, and determine which is the best.</font>

## Step 3.5: extrapolation analysis

All "cold" models fit the data very well, it makes very little difference the one we chose. This statement is no longer correct when we consider extrapolation: the further we are from the data, the more each fit will diverge from each other.

<font color='red'>Task: Plot the data and the 4 "cold" EOS formulations on the same plot, but in a way that the fit goes beyond the data range (e.g. two time higher than the highest measured density). Briefly describe your observations.</font>

## Step 4: add temperature dependence

<font color='red'>Using the best "cold" model, add temperature dependence following the Debye model. Minimal code is provided for the syntax to fit a function of 2 variables ($\rho$ and $T$).</font>

In [ ]:
def func_eos_full(input,rho_0,K_0,alpha):
    '''
    Here, 'input' is a tuple/array of dimension 2xN,
    with column 0 being density and column 1 being temperature.
    We unpack both columns into separates 1D arrays fo size N.
    '''
    rho, T = input
    P_cold = func_BM2(rho,rho_0,K_0)
    P = P_cold + alpha*T
    return P

# -------------------------------------------------------
# 1) Initial guess
# -------------------------------------------------------

rho_0_init =
K_0_init =
alpha_init =

p_init = [
]

# -------------------------------------------------------
# 2) Fit
# -------------------------------------------------------

# Remember: the input is an array of size 2xN, 
# where columns 0 and 1 are density and temperature
x = (pyrolite_data["rho-gcc"],pyrolite_data["T"])
y = pyrolite_data["P-GPa"]
yerr = pyrolite_data["dP-GPa"]

param_opt_full, pcov_full = curve_fit(
    func_eos_full,
    x,
    y,
    p0=p_init,
    sigma=yerr,
    absolute_sigma=True,
    maxfev=10000,
)

param_err_full = np.sqrt(np.diag(pcov_full))

# -------------------------------------------------------
# 3) Print
# -------------------------------------------------------
param_names_full = ["rho_0", "K_0", "alpha"]

param_df_full = pd.DataFrame({
    "Parameter": param_names_full,
    "Initial guess": p_init,
    "Best fit": param_opt_full,
    "1σ uncertainty": param_err_full
})

display(param_df_full)

# -------------------------------------------------------
# 3) Plot
# -------------------------------------------------------
# The slightly tricky part is that we don't have the same
# number of points for each temperature, and
# the temperature fluctuates within +- 20 K.

# residuals computed at data points
residuals_full = y - func_eos_full(x, *param_opt_full)

# ---- plotting ----
# The little trick is that each temperature has a different number of points,
# so we need to define semi-manually the range to plot the fitted model
# at different temperatures.
T_array = np.array([2e3,3e3,4e4,5e3])
rho_model = np.linspace(x[0].min(),x[0].max(),500)
for temp in T_array:
    temp_model = np.full_like(rho_model,temp)
    P_model = func_eos_full((rho_model,temp_model), *param_opt_full)
    plt.plot(rho_model,P_model,c="teal")

plt_scat_error(pyrolite_data["rho-gcc"],
               pyrolite_data["P-GPa"],
               pyrolite_data["dP-GPa"],
               pyrolite_data["T"])

plt.xlabel(r"Density [g.cm$^{-3}$]")
plt.ylabel("Pressure [GPa]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(6,4)
plt.colorbar(label="Temperature [K]")
plt.show()

# ---- Residuals ----
plt_scat_error(pyrolite_data["rho-gcc"],
               residuals_full,
               pyrolite_data["dP-GPa"],
               pyrolite_data["T"])
# plt.errorbar(x[0],residuals_full,yerr=yerr,marker="+",c="orange",ls="")
plt.xlabel(r"Density [g.cm$^{-3}$]")
plt.ylabel("Pressure [GPa]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(6,4)
plt.colorbar(label="Temperature [K]")
plt.show()

## Step 5: Comparison with the literature

<font color='red'>Task: Read the following papers, and compare the thermoelastic parameters that you obtained with your work to the parameters found in the literature.</font>
- Solomatova & Caracas 2021: https://ui.adsabs.harvard.edu/abs/2021JGRB..12621045S/abstract
- Boley + 2023: https://ui.adsabs.harvard.edu/abs/2023ApJ...954..202B/abstract